In [1]:
import os

import pandas as pd
import numpy as np
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("HIP / ROCm detected device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}:", torch.cuda.get_device_name(i))
else:
    print("No GPU visible to PyTorch – check AMD/ROCm setup.")

PyTorch version: 2.10.0+rocm7.2.4.git3d3aa833
CUDA available: True
HIP / ROCm detected device count: 1
Device 0: AMD Radeon Graphics


In [10]:
import pandas as pd

DATA_PATH = "global_startup_success_dataset.csv"  # jeśli plik jest w tym samym folderze co notatnik
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (5000, 15)


,Startup Name,Founded Year,Country,Industry,Funding Stage,Total Funding ($M),Number of Employees,Annual Revenue ($M),Valuation ($B),Success Score,Acquired?,IPO?,Customer Base (Millions),Tech Stack,Social Media Followers
0,Startup_1,2009,Canada,Healthcare,Series A,269,3047,104,46.11,5,No,No,43,"Java, Spring",4158814
1,Startup_2,2004,UK,Healthcare,IPO,40,630,431,33.04,1,No,Yes,64,"Node.js, React",4063014
2,Startup_3,2018,USA,Healthcare,Seed,399,2475,375,15.79,8,No,No,74,"PHP, Laravel",3449855
3,Startup_4,2014,France,Tech,Seed,404,1011,907,17.12,7,Yes,Yes,26,"Python, AI",630421
4,Startup_5,2006,Japan,Energy,Series C,419,3917,280,4.39,6,Yes,Yes,30,"Node.js, React",365956


In [11]:
df.columns.tolist()

['Startup Name',
 'Founded Year',
 'Country',
 'Industry',
 'Funding Stage',
 'Total Funding ($M)',
 'Number of Employees',
 'Annual Revenue ($M)',
 'Valuation ($B)',
 'Success Score',
 'Acquired?',
 'IPO?',
 'Customer Base (Millions)',
 'Tech Stack',
 'Social Media Followers']

In [14]:
def row_to_description(row) -> str:
    name = row.get("startup_name", "")
    industry = row.get("industry", "")
    country = row.get("country", "")
    city = row.get("city", "")
    status = row.get("status", "")
    founded_year = row.get("founded_year", "")
    total_funding = row.get("total_funding_usd", "")

    parts = []
    if name:
        parts.append(f"{name} is a startup")
    else:
        parts.append("This is a startup")

    if industry and isinstance(industry, str):
        parts.append(f"operating in the {industry} industry")

    if city and isinstance(city, str):
        parts.append(f"based in {city}")
    elif country and isinstance(country, str):
        parts.append(f"based in {country}")

    sentence = " ".join(parts) + "."
    extra = []

    if founded_year and not pd.isna(founded_year):
        extra.append(f"It was founded in {int(founded_year)}.")

    if total_funding and not pd.isna(total_funding):
        extra.append(f"It has raised approximately {total_funding} USD in funding.")

    if status and isinstance(status, str):
        extra.append(f"Its current status is: {status}.")

    return " ".join([sentence] + extra)


df["startup_description"] = df.apply(row_to_description, axis=1)
df[["Startup Name", "startup_description"]].head(10)

,Startup Name,startup_description
0,Startup_1,This is a startup.
1,Startup_2,This is a startup.
2,Startup_3,This is a startup.
3,Startup_4,This is a startup.
4,Startup_5,This is a startup.
5,Startup_6,This is a startup.
6,Startup_7,This is a startup.
7,Startup_8,This is a startup.
8,Startup_9,This is a startup.
9,Startup_10,This is a startup.


In [16]:
def build_unicornforge_prompt(desc: str) -> str:
    return f"""You are an experienced startup advisor and hackathon mentor.

Based on the following real-world startup profile, generate a structured startup brief with 10 sections:
1. Project name
2. One-sentence pitch
3. Problem
4. Solution
5. Target market
6. MVP scope
7. Key features
8. Demo scenario
9. Business model
10. Why this startup could succeed

Startup profile:
{desc}
"""


df["uf_prompt"] = df["startup_description"].apply(build_unicornforge_prompt)
df[["Startup Name", "uf_prompt"]].head(3)

,Startup Name,uf_prompt
0,Startup_1,You are an experienced startup advisor and hac...
1,Startup_2,You are an experienced startup advisor and hac...
2,Startup_3,You are an experienced startup advisor and hac...


In [18]:
df.columns.tolist()

['Startup Name',
 'Founded Year',
 'Country',
 'Industry',
 'Funding Stage',
 'Total Funding ($M)',
 'Number of Employees',
 'Annual Revenue ($M)',
 'Valuation ($B)',
 'Success Score',
 'Acquired?',
 'IPO?',
 'Customer Base (Millions)',
 'Tech Stack',
 'Social Media Followers',
 'startup_description',
 'uf_prompt']

In [19]:
import pandas as pd

def row_to_description(row) -> str:
    name = row.get("Startup Name", "")
    industry = row.get("Industry", "")
    country = row.get("Country", "")
    status = row.get("Funding Stage", "")
    founded_year = row.get("Founded Year", "")
    total_funding = row.get("Total Funding ($M)", "")
    employees = row.get("Number of Employees", "")
    revenue = row.get("Annual Revenue ($M)", "")
    valuation = row.get("Valuation ($B)", "")
    customers = row.get("Customer Base (Millions)", "")
    tech_stack = row.get("Tech Stack", "")

    parts = []
    if isinstance(name, str) and name:
        parts.append(f"{name} is a startup")
    else:
        parts.append("This startup")

    if isinstance(industry, str) and industry:
        parts.append(f"operating in the {industry} industry")

    if isinstance(country, str) and country:
        parts.append(f"based in {country}")

    sentence = " ".join(parts) + "."

    extra = []

    if pd.notna(founded_year) and founded_year != "":
        extra.append(f"It was founded in {int(founded_year)}.")

    if pd.notna(total_funding) and total_funding != "":
        extra.append(f"It has raised around {total_funding} million USD in funding.")

    if pd.notna(employees) and employees != "":
        extra.append(f"It employs approximately {employees} people.")

    if pd.notna(revenue) and revenue != "":
        extra.append(f"It generates about {revenue} million USD in annual revenue.")

    if pd.notna(valuation) and valuation != "":
        extra.append(f"Its valuation is estimated at {valuation} billion USD.")

    if pd.notna(customers) and customers != "":
        extra.append(f"It serves roughly {customers} million customers.")

    if isinstance(tech_stack, str) and tech_stack:
        extra.append(f"It uses the following tech stack: {tech_stack}.")

    if isinstance(status, str) and status:
        extra.append(f"Its current funding stage is: {status}.")

    return " ".join([sentence] + extra)


df["startup_description"] = df.apply(row_to_description, axis=1)
df[["Startup Name", "startup_description"]].head(10)

,Startup Name,startup_description
0,Startup_1,Startup_1 is a startup operating in the Health...
1,Startup_2,Startup_2 is a startup operating in the Health...
2,Startup_3,Startup_3 is a startup operating in the Health...
3,Startup_4,Startup_4 is a startup operating in the Tech i...
4,Startup_5,Startup_5 is a startup operating in the Energy...
5,Startup_6,Startup_6 is a startup operating in the FinTec...
6,Startup_7,Startup_7 is a startup operating in the Gaming...
7,Startup_8,Startup_8 is a startup operating in the EdTech...
8,Startup_9,Startup_9 is a startup operating in the AI ind...
9,Startup_10,Startup_10 is a startup operating in the E-com...


In [20]:
def build_unicornforge_prompt(desc: str) -> str:
    return f"""You are an experienced startup advisor and hackathon mentor.

Based on the following real-world startup profile, generate a structured startup brief with 10 sections:
1. Project name
2. One-sentence pitch
3. Problem
4. Solution
5. Target market
6. MVP scope
7. Key features
8. Demo scenario
9. Business model
10. Why this startup could succeed

Startup profile:
{desc}
"""


df["uf_prompt"] = df["startup_description"].apply(build_unicornforge_prompt)
df[["Startup Name", "uf_prompt"]].head(3)

,Startup Name,uf_prompt
0,Startup_1,You are an experienced startup advisor and hac...
1,Startup_2,You are an experienced startup advisor and hac...
2,Startup_3,You are an experienced startup advisor and hac...


In [25]:
%pip install scikit-learn

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [26]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

print("Torch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_PATH = "global_startup_success_dataset.csv"  # dostosuj, jeśli plik jest gdzie indziej

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Torch version: 2.10.0+rocm7.2.4.git3d3aa833
Using device: cuda
Shape: (5000, 15)


,Startup Name,Founded Year,Country,Industry,Funding Stage,Total Funding ($M),Number of Employees,Annual Revenue ($M),Valuation ($B),Success Score,Acquired?,IPO?,Customer Base (Millions),Tech Stack,Social Media Followers
0,Startup_1,2009,Canada,Healthcare,Series A,269,3047,104,46.11,5,No,No,43,"Java, Spring",4158814
1,Startup_2,2004,UK,Healthcare,IPO,40,630,431,33.04,1,No,Yes,64,"Node.js, React",4063014
2,Startup_3,2018,USA,Healthcare,Seed,399,2475,375,15.79,8,No,No,74,"PHP, Laravel",3449855
3,Startup_4,2014,France,Tech,Seed,404,1011,907,17.12,7,Yes,Yes,26,"Python, AI",630421
4,Startup_5,2006,Japan,Energy,Series C,419,3917,280,4.39,6,Yes,Yes,30,"Node.js, React",365956


In [27]:
target_col = "Success Score"

cat_cols = ["Country", "Industry", "Funding Stage", "Tech Stack"]
num_cols = [
    "Founded Year",
    "Total Funding ($M)",
    "Number of Employees",
    "Annual Revenue ($M)",
    "Customer Base (Millions)",
]

used_cols = cat_cols + num_cols + [target_col]
df = df[used_cols].dropna().reset_index(drop=True)
df.head()

,Country,Industry,Funding Stage,Tech Stack,Founded Year,Total Funding ($M),Number of Employees,Annual Revenue ($M),Customer Base (Millions),Success Score
0,Canada,Healthcare,Series A,"Java, Spring",2009,269,3047,104,43,5
1,UK,Healthcare,IPO,"Node.js, React",2004,40,630,431,64,1
2,USA,Healthcare,Seed,"PHP, Laravel",2018,399,2475,375,74,8
3,France,Tech,Seed,"Python, AI",2014,404,1011,907,26,7
4,Japan,Energy,Series C,"Node.js, React",2006,419,3917,280,30,6


In [28]:
# One-hot dla kolumn kategorycznych
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Zapamiętaj nazwy kolumn numerycznych po one-hot
all_cols = list(df_enc.columns)
all_cols.remove(target_col)

X = df_enc[all_cols].astype("float32")
y = df_enc[target_col].astype("float32")

# Standaryzacja numerycznych (dla stabilniejszego treningu)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

X.shape, y.shape

((5000, 31), (5000,))

In [29]:
X_train, X_val, y_train, y_val = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42
)

X_train.shape, X_val.shape

((4000, 31), (1000, 31))

In [30]:
class StartupsDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.from_numpy(x).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


train_ds = StartupsDataset(X_train, y_train)
val_ds = StartupsDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)

In [31]:
input_dim = X_train.shape[1]

class SuccessScoreMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),   # regresja: przewidujemy 1 liczbę
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = SuccessScoreMLP(input_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model

SuccessScoreMLP(
  (net): Sequential(
    (0): Linear(in_features=31, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [32]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            bs = xb.size(0)
            total_loss += loss.item() * bs
            n += bs
    return total_loss / n


n_epochs = 20

for epoch in range(1, n_epochs + 1):
    model.train()
    running_loss = 0.0
    n_train = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        running_loss += loss.item() * bs
        n_train += bs

    train_loss = running_loss / n_train
    val_loss = evaluate(model, val_loader)

    print(f"Epoch {epoch:02d} | train MSE: {train_loss:.4f} | val MSE: {val_loss:.4f}")

Epoch 01 | train MSE: 16.3941 | val MSE: 6.9696
Epoch 02 | train MSE: 6.9717 | val MSE: 6.9074
Epoch 03 | train MSE: 6.8617 | val MSE: 6.8437
Epoch 04 | train MSE: 6.7978 | val MSE: 6.8518
Epoch 05 | train MSE: 6.7499 | val MSE: 6.8468
Epoch 06 | train MSE: 6.6998 | val MSE: 6.9268
Epoch 07 | train MSE: 6.6494 | val MSE: 6.8566
Epoch 08 | train MSE: 6.6219 | val MSE: 6.8932
Epoch 09 | train MSE: 6.5869 | val MSE: 6.8831
Epoch 10 | train MSE: 6.5605 | val MSE: 6.9163
Epoch 11 | train MSE: 6.5143 | val MSE: 6.9624
Epoch 12 | train MSE: 6.4776 | val MSE: 6.8928
Epoch 13 | train MSE: 6.4631 | val MSE: 6.8859
Epoch 14 | train MSE: 6.4244 | val MSE: 6.9135
Epoch 15 | train MSE: 6.3921 | val MSE: 6.9037
Epoch 16 | train MSE: 6.3823 | val MSE: 6.9183
Epoch 17 | train MSE: 6.3629 | val MSE: 6.9177
Epoch 18 | train MSE: 6.3279 | val MSE: 6.9478
Epoch 19 | train MSE: 6.3160 | val MSE: 7.1367
Epoch 20 | train MSE: 6.2808 | val MSE: 6.9859


In [33]:
import os
from pathlib import Path
import joblib  # jeśli nie ma, można pominąć i użyć tylko torch.save

out_dir = Path("trained_models/startup_success_mlp")
out_dir.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), out_dir / "model.pt")

# zapis listy kolumn i scaler’a, żeby backend mógł replikować preprocessing
import pickle

with open(out_dir / "metadata.pkl", "wb") as f:
    pickle.dump(
        {
            "feature_columns": all_cols,
            "num_columns": num_cols,
            "scaler": scaler,
        },
        f,
    )

print("Saved to", out_dir)

Saved to trained_models/startup_success_mlp


In [34]:
from pathlib import Path
import pickle
import numpy as np

out_dir = Path("trained_models/startup_success_mlp")
out_dir.mkdir(parents=True, exist_ok=True)

# policz średnie i odchylenia dla num_cols na TRAIN (tak jak przy normalizacji)
means = {}
stds = {}
for col in num_cols:
    vals = X[col].values.astype("float32")
    m = float(vals.mean())
    s = float(vals.std()) if vals.std() > 0 else 1.0
    means[col] = m
    stds[col] = s

metadata = {
    "feature_columns": all_cols,   # kolumny po one‑hot
    "num_columns": num_cols,
    "means": means,
    "stds": stds,
}

with open(out_dir / "metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Saved metadata to", out_dir / "metadata.pkl")

Saved metadata to trained_models/startup_success_mlp/metadata.pkl
